## Maximum Likelihood Estimation. 

### Maximising Likelihood == Minimising Squared Error

Let us start exploring MLE through code, the goal here is to understand everything about the intuition and math behind MLE via code

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sea
from scipy.stats import stats

In [ ]:
class SGDMultipleLinearRegression():

    def __init__(self):
        self.beta_hat = None
        self.intercept_ = None
        self.coef_ = None
        self.lr_ = None
        self.lr_limit_ = None
        self.loss_history_ = None
        self.n_iter_ = None

    def _add_intercept(self, X):
        X = np.asarray(X, float)

        return np.c_[np.ones(X.shape[0]), X]


    def fit_sgd(self, X, y, batch_size=None, lr=None, epochs=1000,tolerance=1e-08, seed=42, shuffle=False, decay_rate=0.0,
                decay_type=None, patience=5):

        """

        batch_size: an int value for creating mini batches out of the whole data
        lr: learning rate if None use 1/max_ev of X (complete)
        epochs: number of iterations we iterate updating our parameters
        tolerance: checks whether the step size is less than this, if yes we have converged our beta hat at the correct place, 
        checks if step btw two losses is less than tolerance if yes, we declare it as converged !!

        in reality: threshold for the absolute difference between consecutive loss values;
        training stops early if change in loss fallss below this value !! 

        seed: for our rng to start at this seed number only
        shuffle: if you wanna shuffle the data or not 
        decay_type = "time", "exponential" and "plateau" or None
        decay_rate : float value controlling the decay speed/factor
        patience: number of epochs to wait without improvement before reducing lr for plateuau decay !!
        """
        X = np.asarray(X, float)
        y = np.asarray(y, float).ravel()

        X = self._add_intercept(X)   # Build the design matrix with intercept.  

        self.beta_hat = np.zeros(X.shape[1]) #initialise intercept and slope to be ZERO: 0

        n = X.shape[0]#number of rows for hessian matrix
        XTX = X.T @ X # make the X matrix Squared..! 
        H = (2/n) * XTX # calculating Hessian Matrix using formula (2/n) * XTX
        #before loop using all of the dataset !! 

        eigen_values = np.linalg.eigvalsh(H) #evalue of whole dataset. 
        max_ev = np.max(eigen_values)   #max from whole dataset. 

        self.lr_ = lr if lr is not None else 1 / max_ev #take lr if given else initialise you own 
        self.lr_limit_ = 2 / max_ev
        initial_lr = self.lr_

        if self.lr_ >= self.lr_limit_:
            print("Warning! lr_ is at or above the stability limit, expect divergence.")

        tolerance = tolerance #tolerance meaning lr must not be less than 1e-08
        self.loss_history_ = []
        self.n_iter_ = epochs

        if batch_size is None: #if batch size isnot given, take the whole array of X
            batch_size = X.shape[0]
        rng = np.random.default_rng(seed) #seed to keep a fixed start okay !!

        ## decay rate:
        best_loss = np.inf
        epochs_without_improvement = 0


        for epoch in range(epochs):
            
            indices = np.arange(len(X))
            if shuffle:
                rng.shuffle(indices)

            y_pred = X @ self.beta_hat #fit the whole complete X with the current beta get the loss and chec is loss is finite or not ! 
            loss = np.mean((y - y_pred)**2) #recording full dataset loss each epoch, before updates. 

            if not np.isfinite(loss):
                print("Diverged at: ", epoch)
                break

            self.loss_history_.append(loss)

            for start_index in range(0, len(X), batch_size): #make batches, find gradient, update 

                batch_indices = indices[start_index:start_index + batch_size]
        
                X_batch = X[batch_indices]
                y_batch = y[batch_indices]
                B = X_batch.shape[0]

            ## real cooking: 
                y_pred = X_batch @ self.beta_hat
                gradient = -(2/B) * X_batch.T @ (y_batch - y_pred) #the gradient points uphill to the highest slope, so we negate it t point to the smallest/lowest slope okay !!
                self.beta_hat = self.beta_hat - self.lr_ * gradient

            self.n_iter_ = epoch + 1


#--------------------------------------------------------------------------------
            if decay_type == "time":
                self.lr_ = initial_lr / (1 + decay_rate * epoch)

            elif decay_type == "exponential":
                self.lr_ = initial_lr * np.exp(-decay_rate * epoch)

            elif decay_type == "plateau":

                if loss < best_loss - tolerance:
                    best_loss = loss 
                    epochs_without_improvement = 0

                else:
                    epochs_without_improvement += 1
                    if epochs_without_improvement >=patience:
                        self.lr_ = self.lr_ * decay_rate
                        epochs_without_improvement = 0

#------------------------------------------------------------------------------------

            if epoch > 0 and abs(self.loss_history_[-2] - self.loss_history_[-1]) < tolerance:
                print("Converged at: ", epoch)
                break

        self.intercept_ = self.beta_hat[0]
        self.coef_ = self.beta_hat[1:]

        return self

    def predict(self, X):
        if self.beta_hat is None:
            raise ValueError(".fit() not called yet, please fit the data")
        X = np.asarray(X, float)
        X = self._add_intercept(X)

        return X @  self.beta_hat